# 🍽️ Local Food Wastage Management System
**Domain:** Food Management | Waste Reduction | Social Good

**Tech Stack:** Python · SQL (SQLite) · Pandas · Matplotlib · Seaborn · Streamlit

---
## Problem Statement
Food wastage is a significant issue — households and restaurants discard surplus food while many people struggle with food insecurity. This notebook analyzes a food-donation dataset, creates an SQLite database, runs 15+ SQL queries, performs EDA, and visualizes insights.

### Datasets
| File | Description | Rows |
|---|---|---|
| `providers_data.csv` | Food providers (restaurants, supermarkets, etc.) | 1000 |
| `receivers_data.csv` | Food receivers (NGOs, shelters, individuals) | 1000 |
| `food_listings_data.csv` | Available food items with quantities & expiry | 1000 |
| `claims_data.csv` | Claims made by receivers on food listings | 1000 |

## 📦 Section 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('All libraries imported successfully ✅')

## 📂 Section 2 — Load Data

In [ ]:
# ── Load all four CSV files ──────────────────────────────────────────────────
providers     = pd.read_csv('providers_data.csv')
receivers     = pd.read_csv('receivers_data.csv')
food_listings = pd.read_csv('food_listings_data.csv')
claims        = pd.read_csv('claims_data.csv')

print(f'providers     : {providers.shape}')
print(f'receivers     : {receivers.shape}')
print(f'food_listings : {food_listings.shape}')
print(f'claims        : {claims.shape}')

## 🔍 Section 3 — Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
print('=== PROVIDERS ===')
print(providers.head(3).to_string())
print()
print('=== RECEIVERS ===')
print(receivers.head(3).to_string())
print()
print('=== FOOD LISTINGS ===')
print(food_listings.head(3).to_string())
print()
print('=== CLAIMS ===')
print(claims.head(3).to_string())

### 3.2 Data Types & Info

In [ ]:
for name, df in [('providers', providers), ('receivers', receivers),
                  ('food_listings', food_listings), ('claims', claims)]:
    print(f'\n--- {name} ---')
    df.info()

### 3.3 Missing Values Check

In [ ]:
# ── Check for nulls in all datasets ─────────────────────────────────────────
for name, df in [('providers', providers), ('receivers', receivers),
                  ('food_listings', food_listings), ('claims', claims)]:
    total_null = df.isnull().sum().sum()
    print(f'{name:<20} → Total missing values: {total_null}')

print('\nNo missing values detected ✅ — Data is clean and ready for analysis.')

### 3.4 Descriptive Statistics

In [ ]:
print('Food Listings – Quantity Statistics')
food_listings['Quantity'].describe().to_frame().T

### 3.5 Data Cleaning & Type Conversion

In [ ]:
# ── Convert date columns ─────────────────────────────────────────────────────
food_listings['Expiry_Date'] = pd.to_datetime(food_listings['Expiry_Date'])
claims['Timestamp']          = pd.to_datetime(claims['Timestamp'])

# ── Add derived features ─────────────────────────────────────────────────────
claims['Month']  = claims['Timestamp'].dt.to_period('M').astype(str)
claims['Hour']   = claims['Timestamp'].dt.hour
claims['Weekday']= claims['Timestamp'].dt.day_name()

print('Date conversion done ✅')
print(claims[['Timestamp','Month','Hour','Weekday']].head())

## 📊 Section 4 — Data Visualizations

### 4.1 Provider Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pc = providers['Type'].value_counts()
bars = ax.bar(pc.index, pc.values,
              color=['#2ecc71','#3498db','#e74c3c','#f39c12'],
              edgecolor='white', linewidth=1.5)
ax.set_title('Distribution of Food Provider Types')
ax.set_xlabel('Provider Type'); ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(int(bar.get_height())), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Claim Status Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
cs = claims['Status'].value_counts()
axes[0].pie(cs.values, labels=cs.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c','#f39c12'],
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Claim Status — Pie')

# Bar chart
bars = axes[1].bar(cs.index, cs.values,
                   color=['#2ecc71','#e74c3c','#f39c12'], edgecolor='white')
axes[1].set_title('Claim Status — Bar')
axes[1].set_ylabel('Count')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 str(int(bar.get_height())), ha='center', fontweight='bold')
plt.suptitle('Food Claim Status Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Food Type & Meal Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ft = food_listings['Food_Type'].value_counts()
axes[0].pie(ft.values, labels=ft.index, autopct='%1.1f%%',
            colors=['#27ae60','#2980b9','#8e44ad'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Food Type Distribution')

mt = food_listings['Meal_Type'].value_counts()
colors_mt = ['#e74c3c','#3498db','#2ecc71','#f39c12']
bars = axes[1].bar(mt.index, mt.values, color=colors_mt, edgecolor='white')
axes[1].set_title('Meal Type Distribution')
axes[1].set_ylabel('Count')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 str(int(bar.get_height())), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4 Top 10 Most Common Food Items

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_food = food_listings['Food_Name'].value_counts().head(10)
bars = ax.barh(top_food.index[::-1], top_food.values[::-1],
               color=sns.color_palette('viridis', 10))
ax.set_title('Top 10 Most Common Food Items')
ax.set_xlabel('Count')
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 Receiver Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
rt = receivers['Type'].value_counts()
bars = ax.bar(rt.index, rt.values,
              color=['#9b59b6','#1abc9c','#e67e22','#3498db'], edgecolor='white')
ax.set_title('Distribution of Receiver Types')
ax.set_xlabel('Receiver Type'); ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            str(int(bar.get_height())), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 Quantity Distribution Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(food_listings['Quantity'], bins=20,
        color='#3498db', edgecolor='white', alpha=0.85)
mean_q = food_listings['Quantity'].mean()
median_q = food_listings['Quantity'].median()
ax.axvline(mean_q, color='red', linestyle='--', linewidth=2,
           label=f'Mean: {mean_q:.1f}')
ax.axvline(median_q, color='green', linestyle='--', linewidth=2,
           label=f'Median: {median_q:.1f}')
ax.set_title('Distribution of Food Quantity Available')
ax.set_xlabel('Quantity'); ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

### 4.7 Monthly Claims by Status

In [ ]:
monthly = claims.groupby(['Month','Status']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
monthly.plot(kind='bar', ax=ax,
             color=['#e74c3c','#2ecc71','#f39c12'], edgecolor='white')
ax.set_title('Monthly Claims by Status')
ax.set_xlabel('Month'); ax.set_ylabel('Number of Claims')
ax.legend(title='Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.8 Top 10 Cities by Number of Providers

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_cities = providers['City'].value_counts().head(10)
bars = ax.barh(top_cities.index[::-1], top_cities.values[::-1],
               color=sns.color_palette('plasma', 10))
ax.set_title('Top 10 Cities by Number of Food Providers')
ax.set_xlabel('Number of Providers')
for bar in bars:
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.9 Food Type vs Meal Type Heatmap

In [ ]:
ct = pd.crosstab(food_listings['Food_Type'], food_listings['Meal_Type'])
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('Food Type vs Meal Type Heatmap')
plt.tight_layout()
plt.show()

## 🗄️ Section 5 — SQLite Database Creation

In [ ]:
# ── Create (or reconnect to) the SQLite database ─────────────────────────────
conn   = sqlite3.connect('food_wastage.db')
cursor = conn.cursor()

# ── Drop tables if they exist (for clean re-run) ─────────────────────────────
cursor.executescript('''
    DROP TABLE IF EXISTS claims;
    DROP TABLE IF EXISTS food_listings;
    DROP TABLE IF EXISTS providers;
    DROP TABLE IF EXISTS receivers;
''')

# ── Create tables ─────────────────────────────────────────────────────────────
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS providers (
        Provider_ID INTEGER PRIMARY KEY,
        Name        TEXT NOT NULL,
        Type        TEXT,
        Address     TEXT,
        City        TEXT,
        Contact     TEXT
    );

    CREATE TABLE IF NOT EXISTS receivers (
        Receiver_ID INTEGER PRIMARY KEY,
        Name        TEXT NOT NULL,
        Type        TEXT,
        City        TEXT,
        Contact     TEXT
    );

    CREATE TABLE IF NOT EXISTS food_listings (
        Food_ID       INTEGER PRIMARY KEY,
        Food_Name     TEXT,
        Quantity      INTEGER,
        Expiry_Date   TEXT,
        Provider_ID   INTEGER,
        Provider_Type TEXT,
        Location      TEXT,
        Food_Type     TEXT,
        Meal_Type     TEXT,
        FOREIGN KEY (Provider_ID) REFERENCES providers(Provider_ID)
    );

    CREATE TABLE IF NOT EXISTS claims (
        Claim_ID    INTEGER PRIMARY KEY,
        Food_ID     INTEGER,
        Receiver_ID INTEGER,
        Status      TEXT,
        Timestamp   TEXT,
        FOREIGN KEY (Food_ID)     REFERENCES food_listings(Food_ID),
        FOREIGN KEY (Receiver_ID) REFERENCES receivers(Receiver_ID)
    );
''')

# ── Ingest data using pandas to_sql ──────────────────────────────────────────
providers.to_sql('providers',     conn, if_exists='append', index=False)
receivers.to_sql('receivers',     conn, if_exists='append', index=False)
food_listings.to_sql('food_listings', conn, if_exists='append', index=False)
# Keep only columns that exist in SQLite table
claims_db = claims[['Claim_ID','Food_ID','Receiver_ID','Status','Timestamp']]
claims_db.to_sql('claims', conn, if_exists='append', index=False)

conn.commit()
print('Database created and data loaded successfully ✅')

In [ ]:
# ── Verify row counts ─────────────────────────────────────────────────────────
for tbl in ['providers','receivers','food_listings','claims']:
    cursor.execute(f'SELECT COUNT(*) FROM {tbl}')
    print(f'{tbl:<20} rows: {cursor.fetchone()[0]}')

## 🧮 Section 6 — SQL Queries & Analysis (15 Queries)

In [ ]:
def run_query(sql, title=''):
    """Helper: run SQL, return DataFrame with printed title."""
    df = pd.read_sql_query(sql, conn)
    if title:
        print(f'\n📌 {title}')
        print('─' * 55)
    return df

In [ ]:
# ── Q1: Number of food providers per city (top 10) ──────────────────────────
q1 = run_query("""
    SELECT City, COUNT(*) AS Provider_Count
    FROM providers
    GROUP BY City
    ORDER BY Provider_Count DESC
    LIMIT 10;
""", 'Q1 — Top 10 Cities by Number of Food Providers')
q1

In [ ]:
# ── Q2: Provider type contributing the most food ────────────────────────────
q2 = run_query("""
    SELECT Provider_Type,
           COUNT(*)        AS Total_Listings,
           SUM(Quantity)   AS Total_Quantity
    FROM food_listings
    GROUP BY Provider_Type
    ORDER BY Total_Quantity DESC;
""", 'Q2 — Provider Type Contribution')
q2

In [ ]:
# ── Q3: Receivers who claimed the most food ─────────────────────────────────
q3 = run_query("""
    SELECT r.Name, r.Type, r.City,
           COUNT(c.Claim_ID) AS Total_Claims
    FROM claims c
    JOIN receivers r ON c.Receiver_ID = r.Receiver_ID
    GROUP BY r.Receiver_ID
    ORDER BY Total_Claims DESC
    LIMIT 10;
""", 'Q3 — Top 10 Receivers by Number of Claims')
q3

In [ ]:
# ── Q4: Total quantity of food available per city ──────────────────────────
q4 = run_query("""
    SELECT Location AS City,
           SUM(Quantity)  AS Total_Quantity,
           COUNT(Food_ID) AS Total_Items
    FROM food_listings
    GROUP BY Location
    ORDER BY Total_Quantity DESC
    LIMIT 10;
""", 'Q4 — Top 10 Cities by Total Food Quantity')
q4

In [ ]:
# ── Q5: Most commonly available food types ─────────────────────────────────
q5 = run_query("""
    SELECT Food_Type,
           COUNT(*) AS Count,
           SUM(Quantity) AS Total_Quantity
    FROM food_listings
    GROUP BY Food_Type
    ORDER BY Count DESC;
""", 'Q5 — Most Common Food Types')
q5

In [ ]:
# ── Q6: Number of food claims per food item (top 10) ──────────────────────
q6 = run_query("""
    SELECT fl.Food_Name,
           COUNT(c.Claim_ID) AS Total_Claims,
           SUM(fl.Quantity)  AS Total_Quantity
    FROM claims c
    JOIN food_listings fl ON c.Food_ID = fl.Food_ID
    GROUP BY fl.Food_Name
    ORDER BY Total_Claims DESC
    LIMIT 10;
""", 'Q6 — Top 10 Food Items by Number of Claims')
q6

In [ ]:
# ── Q7: Provider with the highest number of successful claims ──────────────
q7 = run_query("""
    SELECT p.Name AS Provider_Name,
           p.Type AS Provider_Type,
           p.City,
           COUNT(c.Claim_ID) AS Completed_Claims
    FROM claims c
    JOIN food_listings fl ON c.Food_ID    = fl.Food_ID
    JOIN providers p      ON fl.Provider_ID = p.Provider_ID
    WHERE c.Status = 'Completed'
    GROUP BY p.Provider_ID
    ORDER BY Completed_Claims DESC
    LIMIT 10;
""", 'Q7 — Top 10 Providers by Successful Claims')
q7

In [ ]:
# ── Q8: Percentage of claims by status ────────────────────────────────────
q8 = run_query("""
    SELECT Status,
           COUNT(*) AS Total,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM claims), 2) AS Percentage
    FROM claims
    GROUP BY Status
    ORDER BY Total DESC;
""", 'Q8 — Claim Status Breakdown (%)')
q8

In [ ]:
# ── Q9: Average quantity of food claimed per receiver ─────────────────────
q9 = run_query("""
    SELECT r.Type AS Receiver_Type,
           ROUND(AVG(fl.Quantity), 2) AS Avg_Quantity_Claimed
    FROM claims c
    JOIN receivers r     ON c.Receiver_ID = r.Receiver_ID
    JOIN food_listings fl ON c.Food_ID    = fl.Food_ID
    GROUP BY r.Type
    ORDER BY Avg_Quantity_Claimed DESC;
""", 'Q9 — Average Quantity Claimed per Receiver Type')
q9

In [ ]:
# ── Q10: Most popular meal type claimed ───────────────────────────────────
q10 = run_query("""
    SELECT fl.Meal_Type,
           COUNT(c.Claim_ID) AS Total_Claims,
           ROUND(COUNT(c.Claim_ID) * 100.0 /
                 (SELECT COUNT(*) FROM claims), 2) AS Percentage
    FROM claims c
    JOIN food_listings fl ON c.Food_ID = fl.Food_ID
    GROUP BY fl.Meal_Type
    ORDER BY Total_Claims DESC;
""", 'Q10 — Most Popular Meal Types in Claims')
q10

In [ ]:
# ── Q11: Total food donated per provider (top 10) ─────────────────────────
q11 = run_query("""
    SELECT p.Name AS Provider_Name,
           p.Type,
           p.City,
           SUM(fl.Quantity)   AS Total_Donated,
           COUNT(fl.Food_ID)  AS Total_Listings
    FROM food_listings fl
    JOIN providers p ON fl.Provider_ID = p.Provider_ID
    GROUP BY p.Provider_ID
    ORDER BY Total_Donated DESC
    LIMIT 10;
""", 'Q11 — Top 10 Providers by Total Food Donated')
q11

In [ ]:
# ── Q12: City with highest number of food listings ────────────────────────
q12 = run_query("""
    SELECT Location AS City,
           COUNT(Food_ID) AS Listing_Count
    FROM food_listings
    GROUP BY Location
    ORDER BY Listing_Count DESC
    LIMIT 1;
""", 'Q12 — City with Highest Number of Food Listings')
q12

In [ ]:
# ── Q13: Food items expiring within 7 days ────────────────────────────────
q13 = run_query("""
    SELECT Food_ID, Food_Name, Quantity, Expiry_Date, Location
    FROM food_listings
    WHERE DATE(Expiry_Date) <= DATE('now', '+7 days')
    ORDER BY Expiry_Date ASC
    LIMIT 15;
""", 'Q13 — Food Items Expiring Soon (Next 7 Days)')
q13

In [ ]:
# ── Q14: Providers with no completed claims ───────────────────────────────
q14 = run_query("""
    SELECT p.Provider_ID, p.Name, p.Type, p.City
    FROM providers p
    WHERE p.Provider_ID NOT IN (
        SELECT DISTINCT fl.Provider_ID
        FROM claims c
        JOIN food_listings fl ON c.Food_ID = fl.Food_ID
        WHERE c.Status = 'Completed'
    )
    LIMIT 10;
""", 'Q14 — Providers with No Completed Claims')
q14

In [ ]:
# ── Q15: Receivers and providers in the same city ────────────────────────
q15 = run_query("""
    SELECT p.City,
           COUNT(DISTINCT p.Provider_ID) AS Providers,
           COUNT(DISTINCT r.Receiver_ID) AS Receivers
    FROM providers p
    JOIN receivers r ON p.City = r.City
    GROUP BY p.City
    ORDER BY Providers DESC
    LIMIT 10;
""", 'Q15 — Cities with Both Providers and Receivers')
q15

## 🔄 Section 7 — CRUD Operations

In [ ]:
# ── CREATE: Add a new food listing ────────────────────────────────────────
cursor.execute("""
    INSERT INTO food_listings
        (Food_ID, Food_Name, Quantity, Expiry_Date, Provider_ID,
         Provider_Type, Location, Food_Type, Meal_Type)
    VALUES (9001, 'Fresh Apples', 30, '2025-07-01', 1,
            'Supermarket', 'Bangalore', 'Vegan', 'Snacks')
""")
conn.commit()
print('✅ CREATE — New food listing inserted.')
pd.read_sql_query('SELECT * FROM food_listings WHERE Food_ID = 9001', conn)

In [ ]:
# ── READ: Fetch the newly inserted record ──────────────────────────────────
result = pd.read_sql_query(
    'SELECT * FROM food_listings WHERE Food_ID = 9001', conn)
print('✅ READ — Record fetched:')
result

In [ ]:
# ── UPDATE: Update quantity of the record ──────────────────────────────────
cursor.execute("""
    UPDATE food_listings
    SET Quantity = 50
    WHERE Food_ID = 9001
""")
conn.commit()
print('✅ UPDATE — Quantity changed to 50:')
pd.read_sql_query('SELECT Food_ID, Food_Name, Quantity FROM food_listings WHERE Food_ID = 9001', conn)

In [ ]:
# ── DELETE: Remove the test record ────────────────────────────────────────
cursor.execute('DELETE FROM food_listings WHERE Food_ID = 9001')
conn.commit()
check = pd.read_sql_query('SELECT COUNT(*) AS cnt FROM food_listings WHERE Food_ID = 9001', conn)
print(f'✅ DELETE — Record deleted. Remaining: {check["cnt"][0]}')

## 💡 Section 8 — Key Insights

In [ ]:
# ── Summary Insights ──────────────────────────────────────────────────────
total_quantity = food_listings['Quantity'].sum()
completed_pct  = round(claims[claims['Status']=='Completed'].shape[0] / len(claims) * 100, 1)
top_provider   = providers['Type'].value_counts().idxmax()
top_food_item  = food_listings['Food_Name'].value_counts().idxmax()
top_city       = providers['City'].value_counts().idxmax()

print('=' * 55)
print('       📊  KEY INSIGHTS — FOOD WASTAGE SYSTEM')
print('=' * 55)
print(f'  Total food quantity available : {total_quantity:,} units')
print(f'  Claims successfully completed : {completed_pct}%')
print(f'  Most common provider type     : {top_provider}')
print(f'  Most frequently listed food   : {top_food_item}')
print(f'  City with most providers      : {top_city}')
print(f'  Total unique food items       : {food_listings["Food_Name"].nunique()}')
print(f'  Total cities with providers   : {providers["City"].nunique()}')
print(f'  Total receiver organizations  : {receivers["Type"].nunique()} types')
print('=' * 55)

In [ ]:
# ── Close the database connection ─────────────────────────────────────────
conn.close()
print('Database connection closed ✅')
print('\n🎉 Analysis complete! All 15 SQL queries executed. CRUD operations demonstrated.')